In [34]:
import os
import shutil
import random
import torch
import torchaudio
import torchaudio.transforms as tt

In [44]:
def segment_audio_array(audio_array, segment_length_in_s=20.0, fade_length_in_s=1.0, sample_rate=44100):
    """Segment an audio array into segments of a given length. Apply fade-in and fade-out to the segments.

    Args:
        audio_array (torch.Tensor): The audio array to segment.
        segment_length_in_s (float, optional): The length of each segment in seconds. Defaults to 20.0.
        fade_length_in_s (float, optional): The length of the fade in and out in seconds. Defaults to 1.0.
        sample_rate (int, optional): The sample rate of the audio. Defaults to 44100.

    Returns:
        segments (torch.Tensor): The segmented audio array.
    """
    assert audio_array.dim() == 2, f"Expected 2D tensor, got {audio_array.dim()}D array."

    # Segment the audio array.
    num_channels, audio_length = audio_array.shape
    segment_length = int(sample_rate * segment_length_in_s)
    num_segments = audio_length // segment_length

    segments = torch.zeros((num_segments, num_channels, segment_length))

    for i in range(num_segments):
        start = i * segment_length
        end = (i + 1) * segment_length
        segments[i, :, :] = audio_array[:, start:end]

    # Apply fade-in and fade-out.
    fade_length = int(sample_rate * fade_length_in_s)
    fade_transform = tt.Fade(fade_in_len=fade_length, fade_out_len=fade_length, fade_shape='linear')
    faded_segments = fade_transform(segments)

    return faded_segments

def slice_demand(demand_root, subset='trn'):
    """Load all WAV audio files in `demand_root`, slice them in segments of length 10s and save the resulting audio files."""
    demand_root = os.path.join(demand_root, subset)
    for environment in os.listdir(demand_root):
        environment_path = os.path.join(demand_root, environment, environment[:-4])
        for file_name in os.listdir(environment_path):
            file_path_src = os.path.join(environment_path, file_name)
            audio, sr = torchaudio.load(file_path_src, channels_first=True)
            faded_segments = segment_audio_array(audio, segment_length_in_s=5.0, fade_length_in_s=0.5, sample_rate=sr)
            for i, segment_i in enumerate(faded_segments):
                slices_path = os.path.join(demand_root, environment, 'SLICES')
                os.makedirs(slices_path, exist_ok=True)
                file_path_dst = os.path.join(slices_path, f'{file_name[:-4]}_{i:03}.wav')
                torchaudio.save(file_path_dst, segment_i, sr)

In [45]:
demand_root = r"/home/ovistetom/Documents/Databases_Local/DEMAND/16k"

In [46]:
slice_demand(demand_root, subset='tst')

In [47]:
a = [1,2,3]
b = [4,5,6,7]

In [48]:
zip(a,b)

In [53]:
for i, (j,k) in enumerate(zip(a,b)):
    print(i,j,k)

0 1 4
1 2 5
2 3 6
